# 06 · Do more dimensions reduce score error?

Selection is fixed at random. Compare D ∈ {1, 2, 3, 5, 8} with the same protocol as 05.

1. **Train:** PCA scree plot, and parallel analysis against random Bernoulli matrices.
2. **Validation:** prediction of hidden answers, and score MAE for each D.
3. **Choose D** by random + gp-IRT validation MAE (mean log MAE over k; ties go to the smaller D).
4. **Test:** report every D. Test never picks D.

Parallel analysis shows structure in the responses. It does not prove extra dimensions help
score estimation; steps 2–4 test that.

In [ ]:
import sys

sys.path.insert(0, "../scripts/irt")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from efficbench import ROOT, load
from protocol import make_splits
from run_progression import run_mirt

ks = [10, 20, 50, 100, 200, 500]
seeds = range(20)
out = ROOT / "results/irt/progression/mirt"

Y, models, items = load()
Y = Y.astype(np.float32)
splits = make_splits(Y)
subjects, model_ids, item_ids = items.subject.values, models.model_id.values, items.item_id.values
dims = [1, 2, 3, 5, 8]

# Optional: {D: [one frozen fitted model per split]} to skip fitting.
contexts_by_dim = None

In [ ]:
chosen_dims, _ = run_mirt(
    Y,
    splits,
    subjects,
    model_ids,
    out,
    dims=sorted(contexts_by_dim) if contexts_by_dim else dims,
    contexts_by_dim=contexts_by_dim,
    ks=ks,
    seeds=seeds,
    item_ids=item_ids,
)
print(f"D chosen on validation: {chosen_dims}")

## Dimensionality on train

Averaged over splits. A component is "real" if its eigenvalue beats the 95th percentile of
100 random Bernoulli matrices with the same item means.

In [ ]:
diagnostics = pd.read_csv(out / "dimensionality.csv")
curve = diagnostics.groupby("component")[["variance_explained", "eigenvalue", "null_95"]].mean()

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
left.plot(curve.index, 100 * curve.variance_explained, "o-")
left.set(xlabel="component", ylabel="variance explained (%)", title="Scree plot")
right.plot(curve.index, curve.eigenvalue, "o-", label="responses")
right.plot(curve.index, curve.null_95, "o--", label="Bernoulli null, 95th pct")
right.set(xlabel="component", ylabel="eigenvalue", yscale="log", title="Parallel analysis")
right.legend()
plt.tight_layout()

## Hidden answers on validation

Show each model a random half of the questions, fit θ, and score predictions on the other half
(log loss, Brier, ECE). `item_mean` is a baseline that ignores the model.

In [ ]:
response = pd.read_csv(out / "response_validation.csv")
response.groupby(["n_dims", "predictor"])[["log_loss", "brier", "ece"]].mean()

## Validation MAE and the chosen D

The gp-IRT weights were fit on validation, so this is tuning error, not a result.

In [ ]:
validation = pd.read_csv(out / "validation_predictions.csv")
val_gp = validation[validation.estimator == "gp-IRT"]
display(100 * val_gp.groupby(["k", "n_dims"]).abs_error.mean().unstack("n_dims"))
pd.read_csv(out / "chosen_dimension.csv")

## Test MAE by D

Same random subsets for every D. Watch k = 200 and 500: if extra dimensions help,
the error floor should drop there.

In [ ]:
test = pd.read_csv(out / "test.csv")
curve = 100 * test[test.estimator == "gp-IRT"].groupby(["n_dims", "k"]).mae.mean().unstack("n_dims")
curve.plot(
    marker="o",
    logx=True,
    figsize=(8, 4.5),
    xlabel="k (answers observed)",
    ylabel="test MAE (pp)",
    title=f"Random + gp-IRT (validation chose D={chosen_dims})",
)
plt.tight_layout()
display(curve)

## Paired change vs D=1

Negative ΔMAE means lower error than the reference. The CI bootstraps the five split means (seeds averaged first). The splits overlap, so read it as sensitivity to the split, not as five independent samples.

In [ ]:
paired = pd.read_csv(out / "paired.csv")
paired = paired[paired.method == "random + gp-IRT"].copy()
paired[["delta_mae", "ci_low", "ci_high"]] *= 100
paired[["n_dims", "k", "delta_mae", "ci_low", "ci_high", "splits_won", "n_splits"]]

Diagnostics follow [Krumdick et al. (2026)](https://arxiv.org/abs/2604.01418).
The chosen D is used in 07.